# Notebook consolidation record

This notebook records the verified migration of historical notebook copies into a ZIP archive, restores matching diagnostic outputs to canonical notebooks, and runs the focused regression tests. The archive preserves old results and receipts; current stage outputs are saved in `experiments/notebooks/`.


In [1]:
from pathlib import Path
from experiments.scripts.consolidate_notebooks import consolidate, ROOT, ARCHIVE, INDEX
result = consolidate()
print(f'Archived {len(result["entries"])} historical notebook/receipt files')
print(f'Archive: {ARCHIVE}')
print(f'Index: {INDEX}')
for item in result['diagnostic_migrations']:
    print(f'{item["canonical"]}: {item["outputs"]} saved output blocks')
outside = [p for p in (ROOT / 'experiments').rglob('*.ipynb') if not p.is_relative_to(ROOT / 'experiments/notebooks')]
assert not outside, f'Notebook files remain outside canonical tree: {outside}'
print('All runnable .ipynb files are under experiments/notebooks')


Archived 66 historical notebook/receipt files
Archive: C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\notebook_history_20260926.zip
Index: C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\notebook_history_20260926.manifest.json
experiments/notebooks/diagnostics/dataset_readiness.ipynb: 55 saved output blocks
experiments/notebooks/diagnostics/phase12_evidence_repair.ipynb: 101 saved output blocks
experiments/notebooks/diagnostics/phase12_qualification_supplement.ipynb: 5 saved output blocks


All runnable .ipynb files are under experiments/notebooks


In [2]:
import subprocess, sys
from experiments.scripts.generate_refactored_notebook import generate_all
generation_code, regenerated = generate_all()
print(f'Regenerated {len(regenerated)} stage notebooks after source change')
assert generation_code == 0
tests = ['tests/test_notebook_executor.py', 'tests/test_notebook_generator.py', 'tests/test_phase1_priority_runners.py', 'tests/test_notebook_consolidation.py']
command = [sys.executable, '-m', 'pytest', '-q', *tests]
print('Running:', ' '.join(command), flush=True)
completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=1200)
print(completed.stdout[-15000:])
print(completed.stderr[-4000:])
assert completed.returncode == 0, f'Focused tests failed: {completed.returncode}'


Regenerated 0 stage notebooks after source change


Running: C:\Users\hung\AppData\Local\Programs\Python\Python311\python.exe -m pytest -q tests/test_notebook_executor.py tests/test_notebook_generator.py tests/test_phase1_priority_runners.py tests/test_notebook_consolidation.py


...................................................                      [100%]
51 passed in 27.86s


